#### Train model HYBRID — giá cơ bản × hệ số nhân — 3 thuật toán

Hybrid = tách `giá cuối = giá cơ bản × hệ số nhân`, train riêng **model giá cơ bản** (file này), rồi
nhân với dự đoán từ **model hệ số nhân đã train ở `train_heso.ipynb`** (nạp lại từ `../<algo>/heso.joblib`,
không train lại). So sánh hybrid với Hướng 1 (giá cuối trực tiếp, từ `pred_gia.parquet`).

> ⚠️ **Cần chạy `train_heso.ipynb` trước** để có sẵn `../HistGB|LightGBM|XGBoost/heso.joblib`.

Các bước: (1) model là gì · (2) setup ban đầu · (3) huấn luyện giá cơ bản theo tháng × thuật toán ·
(4) nạp model hệ số nhân đã lưu, ghép hybrid · (5) so sánh Hybrid vs Hướng 1 · (6) lưu model + dự đoán.

**1. Model huấn luyện là gì**

**Target:** `base_price = target_shown_price / target_shown_multiplier` (giá trước khi nhân surge), log-transform.
Cùng 3 thuật toán: HistGB, LightGBM, XGBoost.

In [2]:
import warnings, time, sys
from pathlib import Path
sys.path.insert(0, "..")
import numpy as np, pandas as pd
import joblib
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from _common_train import CAT, B_NUM, prep, ALGOS, metrics
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

PREP = Path("../../data/hcm_train_ready.parquet")
assert PREP.exists(), "Chua co hcm_train_ready.parquet -> chay chuan_bi_du_lieu.ipynb truoc!"
from _common_train import M_NUM
COLS = list(dict.fromkeys(CAT + B_NUM + M_NUM + ["target_shown_price", "target_shown_multiplier",
        "latest_observed_price", "latest_observed_multiplier", "evaluation_month", "split"]))
COLS = [c for c in COLS if c != "latest_observed_base"]  # cot nay tu tinh sau, khong co san trong parquet
df = pd.read_parquet(PREP, columns=COLS)
df["base_price"] = df.target_shown_price / df.target_shown_multiplier.clip(lower=0.1)
df["latest_observed_base"] = df.latest_observed_price / df.latest_observed_multiplier.clip(lower=0.1)
print(f"Nap {len(df):,} dong x {len(COLS)} cot | base_price median {df.base_price.median():,.0f} VND")

Nap 6,897,051 dong x 23 cot | base_price median 99,281 VND


**2. Setup ban đầu (siêu tham số 3 thuật toán)**

| Thuat toan | Sieu tham so chinh | Y nghia |
|---|---|---|
| **HistGB** (sklearn) | max_iter=500, learning_rate=0.05, l2=1.0, early_stopping | Cay boosting histogram, xu ly categorical/NaN native, on dinh |
| **LightGBM** | n_estimators=800, learning_rate=0.03, num_leaves=63, subsample=0.8 | Cay boosting leaf-wise, thuong nhanh hon tren du lieu lon |
| **XGBoost** | n_estimators=800, learning_rate=0.03, max_depth=7, tree_method="hist" | Cay boosting level-wise, enable_categorical=True |

Ca 3 dung chung mot bo feature va cach xu ly categorical (dtype "category") -> so sanh cong bang.

In [3]:
TARGET = "base_price"
NUM = B_NUM
LOG = True
FEATS = CAT + NUM
print(f"Target={TARGET} | {len(FEATS)} feature | log-target={LOG}")
for algo, tao in ALGOS.items():
    print(f"  [{algo}] {tao()}")

Target=base_price | 14 feature | log-target=True
  [HistGB] HistGradientBoostingRegressor(categorical_features=['service_name',
                                                    'pickup_location_name',
                                                    'dropoff_location_name',
                                                    'weather_main'],
                              early_stopping=True, l2_regularization=1.0,
                              learning_rate=0.05, max_iter=500,
                              n_iter_no_change=20, random_state=42)
  [LightGBM] LGBMRegressor(colsample_bytree=0.8, learning_rate=0.03, n_estimators=800,
              num_leaves=63, random_state=42, reg_lambda=1.0, subsample=0.8,
              verbose=-1)
  [XGBoost] XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=20,
             enable_categorical=True, eval_metri

**3. Huấn luyện model giá cơ bản (theo tháng, 3 cell riêng — mỗi thuật toán 1 cell)**

Với mỗi thuật toán, với mỗi tháng: train trên `split=train`, dự đoán trên `split=test` của cùng tháng.
LightGBM/XGBoost tách riêng 10% train làm validation nội bộ để in loss + early-stop (không đụng `split=test`).

In [4]:
models = {algo: {} for algo in ALGOS}
tests = {algo: {} for algo in ALGOS}
thangs = sorted(df.evaluation_month.unique())
print("Train theo thang:", thangs)

Train theo thang: ['2026-01', '2026-02', '2026-03']


**3a. HistGB**

`train_score_`/`validation_score_` là điểm nội bộ sklearn (early_stopping tự tách 10% train) — loss = -score.

In [5]:
print("=== HistGB ===")
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()
    m = ALGOS["HistGB"]()
    ytr = np.log(tr[TARGET]) if LOG else tr[TARGET]
    m.fit(prep(tr, NUM), ytr)
    pred = m.predict(prep(te, NUM))
    te["base_pred"] = np.exp(pred) if LOG else pred
    models["HistGB"][th] = m; tests["HistGB"][th] = te
    tr_loss = -m.train_score_[-1]; val_loss = -m.validation_score_[-1]
    print(f"  [{th}] train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  so_cay={m.n_iter_}  | {time.time()-t0:.1f}s")

=== HistGB ===
  [2026-01] train_loss=0.0162  val_loss=0.0162  so_cay=500  | 18.8s
  [2026-02] train_loss=0.0163  val_loss=0.0164  so_cay=500  | 17.3s
  [2026-03] train_loss=0.0163  val_loss=0.0164  so_cay=500  | 18.0s


**3b. LightGBM**

10% validation nội bộ, RMSE train/valid, dừng sớm nếu valid không cải thiện sau 20 vòng.

In [6]:
print("=== LightGBM ===")
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()
    ytr_full = np.log(tr[TARGET]) if LOG else tr[TARGET]
    Xtr, Xval, ytr, yval = train_test_split(prep(tr, NUM), ytr_full, test_size=0.1, random_state=42)
    m = ALGOS["LightGBM"]()
    m.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xval, yval)], eval_names=["train", "valid"],
          eval_metric="rmse", callbacks=[lgb.early_stopping(20, verbose=False)])
    pred = m.predict(prep(te, NUM))
    te["base_pred"] = np.exp(pred) if LOG else pred
    models["LightGBM"][th] = m; tests["LightGBM"][th] = te
    tr_loss = m.evals_result_["train"]["rmse"][-1]; val_loss = m.evals_result_["valid"]["rmse"][-1]
    print(f"  [{th}] train_rmse={tr_loss:.4f}  val_rmse={val_loss:.4f}  so_cay={m.best_iteration_}  | {time.time()-t0:.1f}s")

=== LightGBM ===
  [2026-01] train_rmse=0.1793  val_rmse=0.1805  so_cay=800  | 17.1s
  [2026-02] train_rmse=0.1795  val_rmse=0.1803  so_cay=800  | 15.8s
  [2026-03] train_rmse=0.1793  val_rmse=0.1810  so_cay=800  | 18.1s


**3c. XGBoost**

Tương tự LightGBM: 10% validation nội bộ, RMSE train/valid, `early_stopping_rounds=20`.

In [7]:
print("=== XGBoost ===")
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()
    ytr_full = np.log(tr[TARGET]) if LOG else tr[TARGET]
    Xtr, Xval, ytr, yval = train_test_split(prep(tr, NUM), ytr_full, test_size=0.1, random_state=42)
    m = ALGOS["XGBoost"]()
    m.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xval, yval)], verbose=False)
    pred = m.predict(prep(te, NUM))
    te["base_pred"] = np.exp(pred) if LOG else pred
    models["XGBoost"][th] = m; tests["XGBoost"][th] = te
    ev = m.evals_result()
    tr_loss = ev["validation_0"]["rmse"][-1]; val_loss = ev["validation_1"]["rmse"][-1]
    print(f"  [{th}] train_rmse={tr_loss:.4f}  val_rmse={val_loss:.4f}  so_cay={m.best_iteration}  | {time.time()-t0:.1f}s")

=== XGBoost ===
  [2026-01] train_rmse=0.1784  val_rmse=0.1803  so_cay=798  | 36.9s
  [2026-02] train_rmse=0.1786  val_rmse=0.1801  so_cay=796  | 36.4s
  [2026-03] train_rmse=0.1785  val_rmse=0.1809  so_cay=799  | 37.1s


**4. Nạp model hệ số nhân đã lưu, ghép Hybrid = giá cơ bản × hệ số nhân**

In [8]:
allte = {}
for algo in ALGOS:
    heso_path = f"../{algo}/heso.joblib"
    assert Path(heso_path).exists(), f"Chua co {heso_path} -> chay train_heso.ipynb truoc!"
    heso_models = joblib.load(heso_path)
    from _common_train import M_NUM
    d = pd.concat(tests[algo].values())
    preds_mult = []
    for th in thangs:
        te_th = tests[algo][th]
        mult = heso_models[th].predict(prep(te_th, M_NUM))
        preds_mult.append(pd.Series(mult, index=te_th.index))
    d["mult_pred"] = pd.concat(preds_mult)
    d["hybrid_pred"] = d.base_pred * d.mult_pred
    allte[algo] = d
    print(f"[{algo}] da ghep hybrid: {len(d):,} dong")

[HistGB] da ghep hybrid: 864,360 dong
[LightGBM] da ghep hybrid: 864,360 dong
[XGBoost] da ghep hybrid: 864,360 dong


**5. So sánh Hybrid vs Hướng 1 (giá cuối trực tiếp)**

Nạp `pred_gia.parquet` (từ `train_gia.ipynb`) để so trên cùng test-set. In cả bảng tổng gộp lẫn từng test-set nhỏ theo tháng.

In [9]:
PRED_GIA = Path("../evaluation/pred_gia.parquet")
assert PRED_GIA.exists(), "Chua co pred_gia.parquet -> chay train_gia.ipynb truoc!"
gia_h1 = pd.read_parquet(PRED_GIA)

rows = []
for algo in ALGOS:
    dh = allte[algo]
    d1 = gia_h1[gia_h1.algo==algo]
    for ten, dcol, dval in [("Huong 1 (gia truc tiep)", d1, d1["pred"]),
                             ("Hybrid (co ban x he so)", dh, dh["hybrid_pred"])]:
        mk = metrics(dcol["target_shown_price"], dval)
        rows.append({"Thuat toan": algo, "Phuong phap": ten, "Test-set": "TAT CA", "n": len(dcol), **mk})
        for th in thangs:
            dt = dcol[dcol.evaluation_month==th]; dv = dval[dcol.evaluation_month==th]
            mk_t = metrics(dt["target_shown_price"], dv)
            rows.append({"Thuat toan": algo, "Phuong phap": ten, "Test-set": th, "n": len(dt), **mk_t})
bang = pd.DataFrame(rows).round(2)
print("TONG GOP (Huong 1 vs Hybrid, tung thuat toan):")
display(bang[bang["Test-set"]=="TAT CA"])
print("\nTung test-set nho theo thang:")
display(bang[bang["Test-set"]!="TAT CA"])

TONG GOP (Huong 1 vs Hybrid, tung thuat toan):


,Thuat toan,Phuong phap,Test-set,n,MAE,RMSE,R2,MAPE
0,HistGB,Huong 1 (gia truc tiep),TAT CA,864360,18834.01,25644.75,0.70,15.36
4,HistGB,Hybrid (co ban x he so),TAT CA,864360,18047.97,24533.96,0.73,14.74
8,LightGBM,Huong 1 (gia truc tiep),TAT CA,864360,18809.03,25607.87,0.71,15.34
12,LightGBM,Hybrid (co ban x he so),TAT CA,864360,18058.76,24548.54,0.73,14.75
16,XGBoost,Huong 1 (gia truc tiep),TAT CA,864360,18806.72,25602.59,0.71,15.34
20,XGBoost,Hybrid (co ban x he so),TAT CA,864360,18065.93,24568.27,0.73,14.76



Tung test-set nho theo thang:


,Thuat toan,Phuong phap,Test-set,n,MAE,RMSE,R2,MAPE
1,HistGB,Huong 1 (gia truc tiep),2026-01,315360,18622.75,25327.14,0.71,15.44
2,HistGB,Huong 1 (gia truc tiep),2026-02,234632,18700.76,25422.74,0.71,15.34
3,HistGB,Huong 1 (gia truc tiep),2026-03,314368,19145.39,26121.93,0.70,15.29
5,HistGB,Hybrid (co ban x he so),2026-01,315360,17863.11,24331.19,0.73,14.78
6,HistGB,Hybrid (co ban x he so),2026-02,234632,17913.66,24288.23,0.73,14.73
7,HistGB,Hybrid (co ban x he so),2026-03,314368,18333.65,24916.03,0.73,14.72
9,LightGBM,Huong 1 (gia truc tiep),2026-01,315360,18605.25,25306.00,0.71,15.42
10,LightGBM,Huong 1 (gia truc tiep),2026-02,234632,18670.77,25372.77,0.71,15.32
11,LightGBM,Huong 1 (gia truc tiep),2026-03,314368,19116.64,26079.23,0.70,15.27
13,LightGBM,Hybrid (co ban x he so),2026-01,315360,17880.00,24349.61,0.73,14.79


**6. Lưu model giá cơ bản + dự đoán hybrid**

In [10]:
for algo in ALGOS:
    joblib.dump(models[algo], f"../{algo}/hybrid_base.joblib")
    print(f"Da luu ../{algo}/hybrid_base.joblib")

pred_out = pd.concat([allte[algo].assign(algo=algo) for algo in ALGOS], ignore_index=True)
Path("../evaluation").mkdir(exist_ok=True)
pred_out.to_parquet("../evaluation/pred_hybrid.parquet", index=False)
print(f"Da luu ../evaluation/pred_hybrid.parquet ({len(pred_out):,} dong)")
print("=> Mo evaluation/eval_hybrid.ipynb de danh gia chi tiet.")

Da luu ../HistGB/hybrid_base.joblib
Da luu ../LightGBM/hybrid_base.joblib
Da luu ../XGBoost/hybrid_base.joblib
Da luu ../evaluation/pred_hybrid.parquet (2,593,080 dong)
=> Mo evaluation/eval_hybrid.ipynb de danh gia chi tiet.
